In [1]:
# Cella 1: monta Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Cella 2: clona il repo GitHub (prima volta)
# Sostituisci con il tuo username e nome repo
!git clone https://github.com/Fabio-Feruglio/wifi-har-sharp.git /content/wifi-har-sharp

# Oppure se il repo è già clonato, aggiorna solo (sessioni successive)
# %cd /content/wifi-har-sharp
# !git pull

# Cella 3: installa le dipendenze
!pip install -q wandb torchmetrics pyyaml tqdm


# Cella 4: Configurazione mista (Dati RAW locali, Processed e Results su Drive)
import os

# 1. Crea le cartelle di destinazione su Drive se non esistono (tranne raw)
os.makedirs('/content/drive/MyDrive/sharp_project/data/processed', exist_ok=True)
os.makedirs('/content/drive/MyDrive/sharp_project/results', exist_ok=True)

# 2. Crea la cartella del repository escludendo 'raw' che faremo locale
os.makedirs('/content/wifi-har-sharp/data', exist_ok=True)

# 3. IMPORTANTE: Rimuovi l'eventuale symlink 'raw' precedente e crealo come cartella REALE e LOCALE su Colab
!rm -rf /content/wifi-har-sharp/data/raw
os.makedirs('/content/wifi-har-sharp/data/raw', exist_ok=True)

# 4. Crea i symlink verso Drive solo per Processed e Results
# In questo modo i modelli salvati e i dati pronti non andranno persi!
!ln -sfn /content/drive/MyDrive/sharp_project/data/processed \
          /content/wifi-har-sharp/data/processed

!ln -sfn /content/drive/MyDrive/sharp_project/results \
          /content/wifi-har-sharp/results

print("Struttura cartelle configurata correttamente!")

# Cella 4.5: Estrazione dello ZIP sul disco locale temporaneo di Colab
zip_path = "/content/drive/MyDrive/sharp_project/data/raw/dataset.zip" # <-- METTI IL NOME ESATTO DEL TUO ZIP QUI
extract_path = "/content/wifi-har-sharp/data/raw"

print("Estrazione in corso sul disco locale di Colab (può richiedere 2-3 minuti)...")
# Usiamo -q per non stampare migliaia di righe a schermo
!unzip -q "{zip_path}" -d "{extract_path}"
print("Estrazione completata con successo! Ora puoi cercare i file .npy.")

# Cella 5: aggiungi src/ al Python path
import sys
sys.path.insert(0, '/content/wifi-har-sharp/src')

print("Setup completato!")

Mounted at /content/drive
Cloning into '/content/wifi-har-sharp'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 57 (delta 23), reused 37 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 4.54 MiB | 18.81 MiB/s, done.
Resolving deltas: 100% (23/23), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 60.9 MB/s eta 0:00:00
Struttura cartelle configurata correttamente!
Estrazione in corso sul disco locale di Colab (può richiedere 2-3 minuti)...
Estrazione completata con successo! Ora puoi cercare i file .npy.
Setup completato!


In [2]:
import os
import sys

# Spostati nella root del progetto
os.chdir('/content/wifi-har-sharp')

# Aggiungi la cartella src al path di Python per gli import
sys.path.insert(0, '/content/wifi-har-sharp/src')

# Verifica che i dati siano accessibili
print("File presenti in data/raw:", os.listdir('data/raw'))

File presenti in data/raw: ['S1a', 'S3a', 'S6a', 'S2a', 'S7a', 'S2b', 'S4a', 'S1b', 'S5a', 'S1c', 'S6b', 'S4b']


In [3]:
import yaml

with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Iperparametri caricati con successo!")

Iperparametri caricati con successo!


In [4]:

import torch
import torch.nn as nn

class Conv2d_bn(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding="same"):
        super().__init__()

        if padding == "same":
            if isinstance(kernel_size, tuple):
                padding_val = tuple(k // 2 for k in kernel_size)
            else:
                padding_val = kernel_size // 2
        else:
            padding_val = 0

        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding_val)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class Backbone(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()

        # MaxPool2d e padding=1 per allineare le shape (51x171) su tutti i rami
        self.top = nn.MaxPool2d(kernel_size=2, stride=2, padding=1)

        self.mid = Conv2d_bn(in_channels, out_channels=5, kernel_size=2, stride=2, padding="same")

        self.down1 = Conv2d_bn(in_channels, out_channels=3, kernel_size=1, stride=1, padding="same")
        self.down2 = Conv2d_bn(in_channels=3, out_channels=6, kernel_size=2, stride=1, padding="same")
        self.down3 = Conv2d_bn(in_channels=6, out_channels=9, kernel_size=4, stride=2, padding="same")

        self.conv_post_concat = Conv2d_bn(in_channels=15, out_channels=3, kernel_size=1, stride=1, padding="same")

    def forward(self, x):
        out_top = self.top(x)
        out_mid = self.mid(x)

        out_down = self.down1(x)
        out_down = self.down2(out_down)
        out_down = self.down3(out_down)

        # Concatenazione lungo la dimensione dei canali (dim=1)
        x_concat = torch.cat([out_top, out_mid, out_down], dim=1)

        features = self.conv_post_concat(x_concat)
        return features


class SHARPModel(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.backbone = Backbone(in_channels=1)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.2)

        # 3 canali out * 51 altezza * 171 larghezza = 26163 neuroni
        self.dense = nn.Linear(in_features=26163, out_features=num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.flatten(x)
        x = self.dropout(x)
        logits = self.dense(x)
        return logits

In [5]:
# Semplice test di verifica per essere sicuri al 100% che le dimensioni quadrino
if __name__ == "__main__":
    model = SHARPModel(num_classes=7)
    # Simuliamo un batch di 4 spettrogrammi Wi-Fi con dimensione standard (1 canale, 100 velocità, 340 tempo)
    mock_input = torch.randn(4, 1, 100, 340)
    output = model(mock_input)
    print(f"Modello inizializzato con successo!")
    print(f"Forma dell'input:  {mock_input.shape}")
    print(f"Forma dell'output: {output.shape} (Dovrebbe essere [4, 7])")

Modello inizializzato con successo!
Forma dell'input:  torch.Size([4, 1, 100, 340])
Forma dell'output: torch.Size([4, 7]) (Dovrebbe essere [4, 7])


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

# Importiamo i moduli che abbiamo creato insieme
from src.dataset import WiFiDataset
from src.model import SHARPModel

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # Imposta il modello in modalità addestramento (attiva Dropout e BatchNorm)
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in dataloader:
        # Sposta i dati sulla GPU se disponibile
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # 1. Reset dei gradienti
        optimizer.zero_grad()

        # 2. Forward pass (predizione)
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        # 3. Backward pass (calcolo dei gradienti)
        loss.backward()

        # 4. Aggiornamento dei pesi
        optimizer.step()

        # Statistiche temporanee
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:

# SALVATAGGIO E PUSH


!cp "/content/drive/MyDrive/Colab Notebooks/02_baseline_SHARP.ipynb" \
    "/content/wifi-har-sharp/notebooks/"

%cd /content/wifi-har-sharp

!git config --global user.email "fabio.feruglio02@gmail.com"
!git config --global user.name "Fabio-Feruglio"

!git add src/ notebooks/ configs/
!git commit -m "Il tuo messaggio"

# Fix: devi passare le variabili Python a shell in questo modo
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')
USER = "Fabio-Feruglio"
REPO = "wifi-har-sharp"

import subprocess
result = subprocess.run(
    ["git", "push", f"https://{TOKEN}@github.com/{USER}/{REPO}.git", "main"],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

/content
[main 0a20048] Il tuo messaggio
 2 files changed, 95 insertions(+), 51 deletions(-)
 rewrite notebooks/02_baseline_SHARP.ipynb (62%)
 rewrite src/model.py (91%)

To https://github.com/Fabio-Feruglio/wifi-har-sharp.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/Fabio-Feruglio/wifi-har-sharp.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.

